# Notebook 2 — Baseline Anomaly Detection

這份 notebook 接在 Notebook 1 後面。Notebook 1 把原始網路監控欄位轉成特徵；這裡用簡單、可解釋的規則判斷哪些時間點像異常。

本節的核心觀念是：先定義「和最近正常行為相比，現在偏離多少」，再討論模型是否需要更複雜。

我們會使用三種 baseline 方法：

| 方法 | 判斷式 | 初學者解讀 |
|---|---|---|
| Z-score | `(x - mean) / std` | 現在距離最近平均值有幾個標準差。 |
| Robust Z-score | `0.6745 * (x - median) / MAD` | 用中位數和 MAD，比平均值更不容易被極端值帶偏。 |
| Percentile | `x > rolling p95` | 現在是否超過最近歷史中偏高的上界。 |

只要任一方法超過門檻，就把該時間點標為 anomaly flag。

如果你還沒有安裝環境，先閱讀 `aiops-anomaly-detection/getting-started/README.md`。文件中的 project name 和 directory 都是範例，請替換成你自己的名稱與路徑。


## Step 0 — 載入套件、資料、並重建 Notebook 1 的特徵

Notebook 應該盡量能獨立執行，所以這裡會重新讀取資料，並重新計算 Notebook 1 用過的特徵。這樣即使你沒有先執行 Notebook 1，也能從這份 notebook 開始。

執行成功後，你應該看到資料載入完成的訊息。若發生套件錯誤，請先執行 setup script。


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (14, 4)
plt.rcParams['axes.grid'] = True
plt.rcParams['font.size'] = 8
plt.rcParams['axes.titlesize'] = 9
plt.rcParams['axes.labelsize'] = 8
plt.rcParams['xtick.labelsize'] = 7
plt.rcParams['ytick.labelsize'] = 7
plt.rcParams['legend.fontsize'] = 7

ROOT     = Path('../../..')
DATA_CSV = ROOT / 'data/synthetic/synthetic_rrd_metrics.csv'
EVT_CSV  = ROOT / 'data/synthetic/synthetic_event_catalog.csv'

df     = pd.read_csv(DATA_CSV, parse_dates=['timestamp'])
events = pd.read_csv(EVT_CSV,  parse_dates=['start_time', 'end_time'])
df     = df.sort_values(['port_id', 'timestamp']).reset_index(drop=True)

# 衍生特徵（同 Notebook 1）
def safe_div(a, b):
    return np.where(b > 0, a / b, 0.0)

df['traffic_total']   = df['INOCTETS']       + df['OUTOCTETS']
df['ucast_total']     = df['INUCASTPKTS']    + df['OUTUCASTPKTS']
df['errors_total']    = df['INERRORS']        + df['OUTERRORS']
df['discards_total']  = df['INDISCARDS']      + df['OUTDISCARDS']
df['broadcast_total'] = df['INBROADCASTPKTS'] + df['OUTBROADCASTPKTS']
df['multicast_total'] = df['INMULTICASTPKTS'] + df['OUTMULTICASTPKTS']
df['avg_packet_size'] = safe_div(df['traffic_total'],   df['ucast_total'])
df['error_rate']      = safe_div(df['errors_total'],    df['ucast_total'])
df['discard_rate']    = safe_div(df['discards_total'],  df['ucast_total'])
df['broadcast_ratio'] = safe_div(df['broadcast_total'], df['ucast_total'])
df['multicast_ratio'] = safe_div(df['multicast_total'], df['ucast_total'])

print('資料載入完成，筆數:', len(df))

## Step 1 — 為每個 port 計算 rolling baseline

不同 port 的流量大小可能差很多，所以不能把所有 port 混在一起算平均。這裡用 `groupby('port_id')`，讓每個 port 都有自己的 baseline。

每個監控維度都會得到五個統計欄位：

| 後綴 | 意義 |
|---|---|
| `_mean` | 最近 1 小時平均。 |
| `_std` | 最近 1 小時標準差。 |
| `_median` | 最近 1 小時中位數。 |
| `_mad` | 最近 6 小時 median absolute deviation。 |
| `_p95` | 最近 6 小時第 95 百分位。 |

這些欄位不是最終答案，只是後面計分的參考線。


In [ ]:
# 要偵測的欄位
WATCH = {
    'traffic_high':   'traffic_total',
    'packets_high':   'ucast_total',
    'errors_high':    'errors_total',
    'discards_high':  'discards_total',
    'broadcast_high': 'broadcast_total',
    'multicast_high': 'multicast_total',
}

def rolling_mad(s):
    med = np.median(s)
    return np.median(np.abs(s - med))

frames = []
for port_id, g in df.groupby('port_id'):
    g = g.set_index('timestamp').sort_index()
    for col in WATCH.values():
        g[f'{col}_mean']   = g[col].rolling('60min',  min_periods=3).mean()
        g[f'{col}_std']    = g[col].rolling('60min',  min_periods=3).std()
        g[f'{col}_median'] = g[col].rolling('60min',  min_periods=3).median()
        g[f'{col}_mad']    = g[col].rolling('360min', min_periods=6).apply(rolling_mad, raw=True)
        g[f'{col}_p95']    = g[col].rolling('360min', min_periods=6).quantile(0.95)
    frames.append(g.reset_index())

df = pd.concat(frames, ignore_index=True)

# 補齊 warm-up 期間的 NaN
stat_cols = [c for c in df.columns if any(
    c.endswith(s) for s in ('_mean','_std','_median','_mad','_p95'))]
df[stat_cols] = df[stat_cols].bfill().fillna(0)

print('Rolling 統計完成')

## Step 2 — 把偏離 baseline 的程度轉成 anomaly flag

這一步會把連續數值轉成 0/1 flag。`1` 代表該維度在該時間點被規則判為異常，`0` 代表沒有被判為異常。

門檻先用保守的預設值：

```text
Z-score > 3.0
Robust Z-score > 4.0
x > rolling p95
```

這不是數學定理。它是工程上常見、容易解釋的起點。後面你應該調整 `Z_THRESH` 和 `RZ_THRESH`，觀察 recall 和 false positive 如何改變。


In [ ]:
Z_THRESH  = 3.0   # 可以調整這個門檻看看效果
RZ_THRESH = 4.0

for flag, col in WATCH.items():
    std_safe = df[f'{col}_std'].replace(0, np.nan)
    mad_safe = df[f'{col}_mad'].replace(0, np.nan)

    z   = ((df[col] - df[f'{col}_mean']) / std_safe).fillna(0)
    rz  = (0.6745 * (df[col] - df[f'{col}_median']) / mad_safe).fillna(0)
    p95 = df[col] > df[f'{col}_p95']

    df[flag] = ((z > Z_THRESH) | (rz > RZ_THRESH) | p95).astype(int)

flag_cols = list(WATCH.keys())
df['alert_count']    = df[flag_cols].sum(axis=1)   # 同時幾個維度異常
df['any_anomaly']    = (df['alert_count'] > 0).astype(int)
df['event_active']   = (df['event_label'] != 'normal').astype(int)

print('Flag 計算完成')
print('異常筆數:', df['any_anomaly'].sum(), '/', len(df))

## Step 3 — 用圖檢查偵測結果

這張圖有三個 panel。請由上往下讀，不要只看紅點。

1. 第一個 panel：原始流量、rolling mean、正常範圍，以及偵測到的紅點。
2. 第二個 panel：不同監控維度的 anomaly flags。這能看出異常是單一欄位，還是多欄位一起出現。
3. 第三個 panel：同時異常的維度數，以及 ground truth 事件區間。

初學者常犯的錯誤是只看「有沒有紅點」。更好的問題是：紅點是否出現在合理的事件附近？正常時間是否也出現太多紅點？


In [ ]:
def mark_events(ax, port_id, events):
    for _, ev in events.iterrows():
        if ev['port_id'] in [port_id, 'MULTI']:
            ax.axvspan(ev['start_time'], ev['end_time'], alpha=0.12, color='red')

In [ ]:
PORT = 'port-id7427'   # 試試看不同 port
COL  = 'traffic_total'

p = df[df['port_id'] == PORT].copy()

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# --- Panel 1: 原始值 + baseline band ---
ax = axes[0]
ax.plot(p['timestamp'], p[COL], label='traffic_total', linewidth=1, alpha=0.8)
ax.plot(p['timestamp'], p[f'{COL}_mean'], label='rolling mean 1h', linewidth=1.5)
upper = p[f'{COL}_mean'] + 3 * p[f'{COL}_std']
lower = (p[f'{COL}_mean'] - 3 * p[f'{COL}_std']).clip(lower=0)
ax.fill_between(p['timestamp'], lower, upper, alpha=0.15, label='mean ± 3σ')

# 紅點：偵測到 traffic_high 的時間點
flagged = p[p['traffic_high'] == 1]
ax.scatter(flagged['timestamp'], flagged[COL], color='red', s=20, zorder=5, label='traffic_high (偵測)')

mark_events(ax, PORT, events)
ax.set_title(f'{PORT} — traffic_total 與偵測結果')
ax.legend(loc='upper left', fontsize=7)

# --- Panel 2: 所有 flag ---
ax = axes[1]
for flag in flag_cols:
    ax.plot(p['timestamp'], p[flag], linewidth=1, label=flag, alpha=0.8)
mark_events(ax, PORT, events)
ax.set_title('所有維度的 Anomaly Flags (0 = 正常, 1 = 異常)')
ax.set_ylim(-0.1, 1.4)
ax.legend(loc='upper left', fontsize=6, ncol=4)

# --- Panel 3: alert count vs ground truth ---
ax = axes[2]
ax.fill_between(p['timestamp'], p['alert_count'], alpha=0.4, label='alert_count（幾個維度同時異常）')
ax.fill_between(p['timestamp'],
                p['event_active'] * p['alert_count'].max().clip(1),
                alpha=0.2, color='red', label='Ground Truth（真實事件）')
mark_events(ax, PORT, events)
ax.set_title('Alert Count vs Ground Truth')
ax.legend(loc='upper left', fontsize=7)

plt.tight_layout()
plt.show()

## Step 4 — 用 recall 表做第一層評估

Recall 的意思是：真實事件發生時，有多少比例被偵測到。

```text
Recall = 被偵測到的事件時間點 / 所有事件時間點
```

表中的 `normal` 不是事件。它代表正常資料被誤判成異常的比例，可把它理解成 false positive rate 的近似讀法。這對維運很重要：偵測率高但正常時間一直告警，系統仍然不可用。


In [ ]:
recall = (
    df.groupby('event_label')
    .agg(
        筆數  = ('timestamp',    'size'),
        偵測到 = ('any_anomaly', 'sum'),
        平均同時異常維度 = ('alert_count', 'mean'),
    )
    .assign(Recall = lambda x: (x['偵測到'] / x['筆數']).round(3))
    .sort_values('Recall', ascending=False)
)
recall

## Step 5 — 跨 port 比較 broadcast storm

Broadcast storm 的特性通常不是單一 port 獨自變化，而是多個 port 在接近的時間一起升高。這就是為什麼 AIOps 不能只看單點，也要看跨設備、跨介面的同步行為。

這一步會把所有 port 的 `broadcast_total` 和 `broadcast_high` 畫在一起。請觀察事件區間內是否有多條線同時升高。


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

# Panel 1: 各 port 的 broadcast_total
for port_id, g in df.groupby('port_id'):
    axes[0].plot(g['timestamp'], g['broadcast_total'], linewidth=1, label=port_id)
axes[0].set_title('所有 Port — broadcast_total')
axes[0].legend(loc='upper left', fontsize=7)

# Panel 2: 各 port 的 broadcast_high flag
for port_id, g in df.groupby('port_id'):
    axes[1].plot(g['timestamp'], g['broadcast_high'], linewidth=1.2, label=port_id, alpha=0.8)

# 標記 Event G
ev_g = events[events['event_type'] == 'broadcast_storm'].iloc[0]
for ax in axes:
    ax.axvspan(ev_g['start_time'], ev_g['end_time'], alpha=0.15, color='red', label='Event G')

axes[1].set_title('所有 Port — broadcast_high Flag (0/1)')
axes[1].set_ylim(-0.1, 1.4)
axes[1].legend(loc='upper left', fontsize=7)

plt.tight_layout()
plt.show()

## 練習與檢查點

完成本 notebook 後，請用自己的話回答下列問題。

1. `large_file_backup` 的 recall 為什麼可能偏低？提示：持續偏高和突然跳升不是同一種異常。
2. `normal` 的 anomaly rate 大約是多少？如果每分鐘都有 false alert，維運人員會遇到什麼問題？
3. 把 `Z_THRESH` 改成 `2.0`，再改成 `4.0`。哪個設定比較敏感？哪個設定比較保守？
4. Broadcast storm 期間，哪些 port 最早出現 `broadcast_high`？這能不能作為 RCA 的線索？

這份 notebook 的結論應該很清楚：baseline 方法簡單、可解釋、部署快，但對緩慢變化或長時間偏高的事件不一定敏感。後續方法要解決的就是這些限制。
